# Kink-aware MLP: weighted fit + repulsion

Та же функция, точки и seed, что в обычном MLP: `Linear(1, 128) → ReLU → Linear(128, 1)`. Все параметры обучаемы, роста и заморозки нет. Исходный выходной bias равен нулю, как у конкурента.

Каждая из 1600 итераций содержит **два последовательных обновления**:
1. Adam обновляет все параметры по weighted MSE. Веса `q = 1 + lambda_fit * rho` отсоединены от графа; нормировка — на `sum(q)`.
2. Изломы `t = -b/w` пересчитываются после Adam. Отдельный SGD обновляет только скрытый слой по сумме Gaussian repulsion для пар `i < j`.

Изломы при `abs(w) < min_abs_weight` и вне интервала исключены из обоих механизмов. Координаты изломов не являются параметрами. Репульсионный градиент проходит через `-b/w`, предобусловливается множителем `w²/(1+t²)` и ограничивается по смещению излома; норма `(w,b)` сохраняется после репульсии. Совпавшие изломы имеют нулевой репульсионный градиент — искусственный шум не добавляется.

Сверху показан **обычный MSE после обоих подшагов**, чтобы сравнивать с другими ноутбуками; снизу — функция и фит. Значения weighted fitting loss и repulsion loss доступны в `kink_experiment.kink_history`. Ширины Gaussian и максимальное смещение заданы в единицах оси x.

In [1]:
from pathlib import Path
import sys

project_root = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file() and (p / "notebooks" / "_shared").is_dir()
)
shared_parent = str(project_root / "notebooks")
if shared_parent not in sys.path:
    sys.path.insert(0, shared_parent)

import torch
from _shared import Config
from _shared.kink_training import KinkTrainingConfig
from _shared.kink_experiment import KinkAwareExperiment
from _shared.visualization import train_live

torch.set_num_threads(1)

In [6]:
config = Config(
    seed=42,
    group_sizes=(512,),
    steps_per_stage=3200,
    learning_rate=0.01,
    n_train=256,
    n_grid=601,
    x_min=-3.0,
    x_max=3.0,
)
training_config = KinkTrainingConfig(
    sigma_fit=0.05,
    lambda_fit=1.0,
    sigma_rep=0.05,
    lr_fit=config.learning_rate,
    lr_rep=0.001,
    min_abs_weight=1e-4,
    max_repulsion_step=0.05,
)
target_fps = 10  # Максимальная частота отрисовки, обучение без пауз.

In [8]:
kink_experiment = KinkAwareExperiment(config, training_config)
await train_live(kink_experiment, target_fps=target_fps)

CancelledError: 